In [1]:
pip install gymnasium

   ---------------------------------------- 0.0/952.1 kB ? eta -:--:--
   --------------------------------------- 952.1/952.1 kB 21.5 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   ---------------------------------------- 2/2 [gymnasium]

Note: you may need to restart the kernel to use updated packages.


In [4]:
import gymnasium as gym

# Create the environment
env = gym.make("CartPole-v1")
state, info = env.reset()

print(f"Initial State: {state}")
# The state is an array of 4 values: [Cart Position, Cart Velocity, Pole Angle, Pole Velocity At Tip]

# Take a random action (0 = push left, 1 = push right)
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)

print(f"Action taken: {action}")
print(f"Next State: {next_state}")
print(f"Reward: {reward}") 

env.close()

Initial State: [0.00282732 0.03919616 0.0169685  0.03814816]
Action taken: 0
Next State: [ 0.00361124 -0.15616496  0.01773146  0.33613616]
Reward: 1.0


In [6]:
pip install torch

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   -- ------------------------------------- 5.8/114.6 MB 30.2 MB/s eta 0:00:04
   ----- ---------------------------------- 14.9/114.6 MB 36.7 MB/s eta 0:00:03
   ----------- ---------------------------- 33.8/114.6 MB 55.3 MB/s eta 0:00:02
   ---------------- ----------------------- 48.2/114.6 MB 57.7 MB/s eta 0:00:02
   ----------------------- ---------------- 67.6/114.6 MB 65.2 MB/s eta 0:00:01
   --------------------------- ------------ 80.0/114.6 MB 63.8 MB/s eta 0:00:01
   ----------------------------- ---------- 84.1/114.6 MB 59.1 MB/s eta 0:00:01
   ------------------------------- -------- 89.4/114.6 MB 53.5 MB/s eta 0:00:01
   --------------------------------- ------ 97.3/114.6 MB 52.0 MB/s eta 0:00:01
   ------------------------------------- - 110.1/114.6 MB 52.7 MB/s eta 0:00:01
   --------------------------------------- 114.6/114.6 MB 49.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use up

In [7]:
import torch
import torch.nn as nn

# Define a simple 3-layer neural network
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)   # Input layer
        self.layer2 = nn.Linear(hidden_size, hidden_size)  # Hidden layer
        self.layer3 = nn.Linear(hidden_size, output_size)  # Output layer
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.layer3(x)  # No activation on output (e.g. for regression or raw logits)
        return x

# CartPole has 4 state inputs and 2 possible actions
input_size = 4
hidden_size = 64
output_size = 2

model = SimpleNet(input_size, hidden_size, output_size)
print(model)

# Test with a sample input
sample_input = torch.tensor([[0.1, -0.05, 0.02, -0.03]], dtype=torch.float32)
output = model(sample_input)
print(f"\nSample input:  {sample_input}")
print(f"Network output (Q-values): {output}")


SimpleNet(
  (layer1): Linear(in_features=4, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=2, bias=True)
  (relu): ReLU()
)

Sample input:  tensor([[ 0.1000, -0.0500,  0.0200, -0.0300]])
Network output (Q-values): tensor([[-0.0119, -0.0229]], grad_fn=<AddmmBackward0>)


In [8]:
import random
from collections import deque

# --- Experience Replay Buffer ---
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(states, dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states, dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


In [9]:
# --- Hyperparameters ---
EPISODES       = 500
BATCH_SIZE     = 64
GAMMA          = 0.99    # discount factor
LR             = 1e-3    # learning rate
EPSILON_START  = 1.0     # start fully random
EPSILON_END    = 0.01    # minimum exploration
EPSILON_DECAY  = 0.995   # multiplicative decay per episode
MIN_BUFFER     = 500     # start training only after this many experiences
TARGET_SCORE   = 500     # CartPole-v1 max steps — considered "solved"

# --- Setup ---
env        = gym.make("CartPole-v1")
policy_net = SimpleNet(input_size=4, hidden_size=64, output_size=2)
optimizer  = torch.optim.Adam(policy_net.parameters(), lr=LR)
loss_fn    = nn.MSELoss()
buffer     = ReplayBuffer()
epsilon    = EPSILON_START

# --- Helper: select action with epsilon-greedy policy ---
def select_action(state, eps):
    if random.random() < eps:
        return env.action_space.sample()                          # explore
    with torch.no_grad():
        state_t = torch.tensor([state], dtype=torch.float32)
        return policy_net(state_t).argmax(dim=1).item()           # exploit

# --- Helper: update network from a sampled mini-batch ---
def train_step():
    if len(buffer) < MIN_BUFFER:
        return None
    states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)

    # Current Q-values for the actions actually taken
    q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Bellman target: r + γ * max_a Q(s', a)  (0 if terminal)
    with torch.no_grad():
        max_next_q = policy_net(next_states).max(dim=1).values
        targets = rewards + GAMMA * max_next_q * (1 - dones)

    loss = loss_fn(q_values, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

# --- Training Loop ---
scores = []
for episode in range(1, EPISODES + 1):
    state, _ = env.reset()
    total_reward = 0

    while True:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buffer.push(state, action, reward, next_state, float(done))
        train_step()

        state = next_state
        total_reward += reward
        if done:
            break

    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    scores.append(total_reward)

    if episode % 50 == 0:
        avg = sum(scores[-50:]) / 50
        print(f"Episode {episode:4d} | Avg score (last 50): {avg:6.1f} | epsilon: {epsilon:.3f}")

    if sum(scores[-100:]) / min(len(scores), 100) >= TARGET_SCORE:
        print(f"\nSolved in {episode} episodes!")
        break

env.close()
print("Training complete.")


C:\Users\valga\AppData\Local\Temp\ipykernel_1664\1383392694.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  state_t = torch.tensor([state], dtype=torch.float32)


Episode   50 | Avg score (last 50):   27.8 | epsilon: 0.778
Episode  100 | Avg score (last 50):   67.5 | epsilon: 0.606
Episode  150 | Avg score (last 50):  106.5 | epsilon: 0.471
Episode  200 | Avg score (last 50):  184.3 | epsilon: 0.367
Episode  250 | Avg score (last 50):   38.1 | epsilon: 0.286
Episode  300 | Avg score (last 50):   11.8 | epsilon: 0.222
Episode  350 | Avg score (last 50):   10.3 | epsilon: 0.173
Episode  400 | Avg score (last 50):   10.1 | epsilon: 0.135
Episode  450 | Avg score (last 50):  153.7 | epsilon: 0.105
Episode  500 | Avg score (last 50):   87.5 | epsilon: 0.082
Training complete.
